In [ ]:
import os
import random
import re
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

from transformers import CLIPProcessor, CLIPModel, DistilBertTokenizerFast, DistilBertModel

In [ ]:
base_path = '/content/drive/MyDrive/FINAL PHISHING DATASET'

phish_csv = os.path.join(base_path, 'Phishing (1).csv')
legit_csv = os.path.join(base_path, 'Legitimate.csv')

phish_img_dir = os.path.join(base_path, 'PHISHING')
legit_img_dir = os.path.join(base_path, 'LEGITIMATE')

phish_df = pd.read_csv(phish_csv)
legit_df = pd.read_csv(legit_csv)

def sort_numerically(file_list):
    return sorted(file_list, key=lambda x: int(''.join(filter(str.isdigit, x)) or 0))

phish_images = sort_numerically(os.listdir(phish_img_dir))
legit_images = sort_numerically(os.listdir(legit_img_dir))


In [ ]:
def map_images_to_urls(df, images, img_dir):
    image_paths = []
    for i in range(len(images)):
        csv_index = i
        if csv_index < len(df):
            image_paths.append(os.path.join(img_dir, images[i]))
        else:
            image_paths.append(os.path.join(img_dir, images[-1]))
    df = df.iloc[:len(image_paths)].copy()
    df['image_path'] = image_paths
    return df

phish_df['label'] = 1
legit_df['label'] = 0

phish_df = map_images_to_urls(phish_df, phish_images, phish_img_dir)
legit_df = map_images_to_urls(legit_df, legit_images, legit_img_dir)

In [ ]:
data = pd.concat([phish_df, legit_df], ignore_index=True)
url_col = 'url' if 'url' in data.columns else data.columns[0]
data['url'] = data[url_col].astype(str).str.strip()

In [ ]:
def extract_url_features(url):
    return [
        len(url),
        url.count('.'),
        url.count('-'),
        url.count('@'),
        sum(c.isdigit() for c in url),
        int(url.startswith('https')),
        int(bool(re.match(r'http[s]?://\d+\.\d+\.\d+\.\d+', url))),
        int(any(w in url for w in ['login','secure','update','verify','signin','bank']))
    ]

url_features = np.array([extract_url_features(u) for u in tqdm(data['url'])])
scaler = StandardScaler()
url_feat_scaled = scaler.fit_transform(url_features)

100%|██████████| 6000/6000 [00:00<00:00, 79526.44it/s]


In [ ]:
class PhishScreenshotDataset(Dataset):
    def __init__(self, df, url_features, image_transform=None, max_url_len=200):
        self.df = df.reset_index(drop=True).copy()
        self.url_feats = url_features
        self.clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        self.tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
        self.max_url_len = max_url_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert("RGB")
        clip_inputs = self.clip_processor(images=image, return_tensors="pt")
        pixel_values = clip_inputs['pixel_values'].squeeze(0)

        tokens = self.tokenizer(row['url'], padding='max_length', truncation=True,
                                max_length=self.max_url_len, return_tensors='pt')
        input_ids = tokens['input_ids'].squeeze(0)
        attention_mask = tokens['attention_mask'].squeeze(0)

        url_feat_tensor = torch.tensor(self.url_feats[idx], dtype=torch.float32)
        label = torch.tensor(row['label'], dtype=torch.long)

        return {
            'pixel_values': pixel_values,
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'url_feats': url_feat_tensor,
            'label': label
        }

In [ ]:
class MultimodalPhishModel(nn.Module):
    def __init__(self, url_feat_dim, proj_dim=256, dropout=0.2, seq_len=200):
        super().__init__()
        self.clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
        for p in self.clip.vision_model.parameters():
            p.requires_grad = False

        self.url_encoder = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.url_feat_proj = nn.Linear(url_feat_dim, 64)
        self.image_proj = nn.Linear(768, 256)
        self.url_text_proj = nn.Linear(768, 256)

        with torch.no_grad():
            dummy_image = torch.zeros(1, 3, 224, 224)
            dummy_input_ids = torch.zeros(1, seq_len, dtype=torch.long)
            dummy_mask = torch.ones(1, seq_len, dtype=torch.long)
            dummy_url_feat = torch.zeros(1, url_feat_dim)

            img_emb = self.image_proj(self.clip.vision_model(dummy_image).last_hidden_state.mean(1))
            url_emb = self.url_text_proj(self.url_encoder(input_ids=dummy_input_ids,
                                                          attention_mask=dummy_mask).last_hidden_state[:, 0, :])
            url_feat_emb = self.url_feat_proj(dummy_url_feat)
            fusion_input_dim = img_emb.shape[1] + url_emb.shape[1] + url_feat_emb.shape[1]

        self.fusion = nn.Sequential(
            nn.Linear(fusion_input_dim, proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(proj_dim, 2)
        )

    def forward(self, pixel_values, input_ids, attention_mask, url_feats):
        clip_out = self.clip.vision_model(pixel_values=pixel_values)
        img_emb = self.image_proj(clip_out.last_hidden_state.mean(1))

        url_out = self.url_encoder(input_ids=input_ids, attention_mask=attention_mask)
        url_emb = self.url_text_proj(url_out.last_hidden_state[:, 0, :])

        url_feat_emb = self.url_feat_proj(url_feats)
        fused = torch.cat([img_emb, url_emb, url_feat_emb], dim=1)
        logits = self.fusion(fused)
        return logits, fused

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, scaler=None):
    model.train()
    losses, all_preds, all_labels = [], [], []

    for batch in tqdm(loader):
        pixel_values = batch['pixel_values'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        url_feats = batch['url_feats'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        if scaler:
            with torch.cuda.amp.autocast():
                logits, _ = model(pixel_values, input_ids, attention_mask, url_feats)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits, _ = model(pixel_values, input_ids, attention_mask, url_feats)
            loss = criterion(logits, labels)     # ✅ FIXED: define loss here
            loss.backward()
            optimizer.step()

        losses.append(loss.item())
        preds = logits.argmax(dim=1).detach().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.detach().cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    return np.mean(losses), acc


In [ ]:
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for batch in loader:
            pixel_values = batch['pixel_values'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            url_feats = batch['url_feats'].to(device)
            labels = batch['label'].to(device)

            logits, _ = model(pixel_values, input_ids, attention_mask, url_feats)
            probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            preds = logits.argmax(dim=1).detach().cpu().numpy()

            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.detach().cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = None

    return {'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1, 'auc': auc}


In [ ]:
data['domain'] = data['url'].apply(lambda u: u.split("://")[-1].split('/')[0])
unique_domains = data['domain'].unique().tolist()
random.seed(42)
random.shuffle(unique_domains)
train_domains = set(unique_domains[:int(0.8 * len(unique_domains))])
val_domains = set(unique_domains[int(0.8 * len(unique_domains)):])

train_df = data[data['domain'].isin(train_domains)].reset_index(drop=True)
val_df = data[data['domain'].isin(val_domains)].reset_index(drop=True)

train_idx = train_df.index.values
val_idx = val_df.index.values
train_url_feats = url_feat_scaled[train_idx]
val_url_feats = url_feat_scaled[val_idx]

train_ds = PhishScreenshotDataset(train_df, train_url_feats)
val_ds = PhishScreenshotDataset(val_df, val_url_feats)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

url_feat_dim = train_url_feats.shape[1]
model = MultimodalPhishModel(url_feat_dim=url_feat_dim).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None

num_epochs = 8
best_val_f1 = 0.0


for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler)
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")

    val_metrics = evaluate(model, val_loader, device)
    print(f"Val Acc: {val_metrics['acc']:.4f} | "
          f"Prec: {val_metrics['prec']:.4f} | "
          f"Rec: {val_metrics['rec']:.4f} | "
          f"F1: {val_metrics['f1']:.4f} | "
          f"AUC: {val_metrics['auc']:.4f}" if val_metrics['auc'] else "AUC: N/A")

    if val_metrics['f1'] > best_val_f1:
        best_val_f1 = val_metrics['f1']
        torch.save(model.state_dict(), "best_multimodal_phish_model.pt")
        print("Best model updated and saved.")


print("\nLoading best model for final evaluation...")
model.load_state_dict(torch.load("best_multimodal_phish_model.pt", map_location=device))
final_metrics = evaluate(model, val_loader, device)
print(f"Final Metrics → Acc: {final_metrics['acc']:.4f}, "
      f"Prec: {final_metrics['prec']:.4f}, "
      f"Rec: {final_metrics['rec']:.4f}, "
      f"F1: {final_metrics['f1']:.4f}, "
      f"AUC: {final_metrics['auc']:.4f}" if final_metrics['auc'] else "AUC: N/A")


Using device: cpu


pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

OSError: Can't load the model for 'openai/clip-vit-base-patch32'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'openai/clip-vit-base-patch32' is the correct path to a directory containing a file named pytorch_model.bin, tf_model.h5, model.ckpt or flax_model.msgpack.